In [ ]:
import gc, json, time, torch, timm
assert torch.cuda.is_available(), 'Select a GPU before running this preflight'
print('torch',torch.__version__,'timm',timm.__version__,'GPU',torch.cuda.get_device_name(0))
"""Shared model architecture for training and offline inference."""
import torch
import torch.nn as nn
import timm

def make_neighbours(x):
    # (B,S,T,H,W) -> (B,S,T,3,H,W) from slices t-1,t,t+1 (edges clamped)
    prev = torch.cat([x[:, :, :1], x[:, :, :-1]], dim=2)
    nxt  = torch.cat([x[:, :, 1:], x[:, :, -1:]], dim=2)
    return torch.stack([prev, x, nxt], dim=3)


class AttnPool(nn.Module):
    # Attention over slices within one plane.
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):               # x (B,T,D)
        a = self.score(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(~mask, torch.finfo(a.dtype).min)
        a = a.softmax(dim=-1)
        return (x * a.unsqueeze(-1)).sum(1), a


class LabelAttnPool(nn.Module):
    # Per-label attention over planes: each finding picks the view it needs.
    def __init__(self, dim, n_labels):
        super().__init__()
        self.score = nn.Linear(dim, n_labels)

    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)
        a = self.score(x)                          # (B,S,L)
        a = a.masked_fill(~mask.unsqueeze(-1), torch.finfo(a.dtype).min)
        a = a.softmax(dim=1)                       # over planes, per label
        z = torch.einsum("bsd,bsl->bld", x, a)     # (B,L,D)
        return z, a


class KneeNet(nn.Module):
    def __init__(self, backbone="tf_efficientnet_b0.ns_jft_in1k", n_out=12, n_planes=3,
                 pretrained=True, drop=0.3, img=224, normalization="legacy",
                 encoder_chunk=64):
        super().__init__()
        kwargs = {"img_size": img} if backbone.startswith("vit_") else {}
        self.enc = timm.create_model(backbone, pretrained=pretrained,
                                     num_classes=0, in_chans=3, **kwargs)
        self.encoder_chunk = encoder_chunk
        config = timm.data.resolve_model_data_config(self.enc)
        mean = config['mean'] if normalization == 'pretrained' else (0.449,)*3
        std = config['std'] if normalization == 'pretrained' else (0.226,)*3
        self.register_buffer('input_mean', torch.tensor(mean).view(1,3,1,1), persistent=False)
        self.register_buffer('input_std', torch.tensor(std).view(1,3,1,1), persistent=False)
        d = self.enc.num_features
        self.slice_pool = AttnPool(d)
        self.plane_emb = nn.Parameter(torch.zeros(n_planes, d))
        self.plane_pool = LabelAttnPool(d, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))
        nn.init.trunc_normal_(self.head_w, std=0.02)

    def forward(self, x, present):
        # autocast entered HERE, not in the training loop: it is thread-local, so a
        # loop-level autocast never reaches a threaded forward. Verified reaching the
        # encoder as float16.
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            B, S, T, H, W = x.shape
            x = x.float().div(255.)   # ImageNet grey stats
            x3 = make_neighbours(x).reshape(B * S * T, 3, H, W)

            x3 = (x3 - self.input_mean) / self.input_std
            # Chunk evaluation to bound memory; training keeps BatchNorm batch statistics intact.
            if not self.training and self.encoder_chunk and len(x3) > self.encoder_chunk:
                features = torch.cat([self.enc(part) for part in x3.split(self.encoder_chunk)])
            else:
                features = self.enc(x3)
            f = features.reshape(B * S, T, -1)
            v, _ = self.slice_pool(f)
            v = v.reshape(B, S, -1) + self.plane_emb.unsqueeze(0)

            m = present.clone()
            m[~m.any(dim=1)] = True                # guard: never all-masked
            z, plane_att = self.plane_pool(v, m)
            z = self.drop(z)
            logits = (z * self.head_w.unsqueeze(0)).sum(-1) + self.head_b
        return logits, plane_att


results=[]
for name,batch in [('tf_efficientnet_b0.ns_jft_in1k',4),('vit_small_patch16_dinov3.lvd1689m',1)]:
    model=None
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        model=KneeNet(name,12,3,pretrained=False,img=336,normalization='pretrained').cuda().train()
        x=torch.randint(0,256,(batch,3,10,336,336),dtype=torch.uint8,device='cuda')
        p=torch.ones((batch,3),dtype=torch.bool,device='cuda')
        target=torch.zeros((batch,12),device='cuda')
        scaler=torch.amp.GradScaler('cuda')
        optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4)
        for i in range(4):
            torch.cuda.synchronize(); started=time.time()
            optimizer.zero_grad(set_to_none=True)
            logits,_=model(x,p)
            loss=torch.nn.functional.binary_cross_entropy_with_logits(logits.float(),target)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            torch.cuda.synchronize()
            print(name,'step',i,'seconds',round(time.time()-started,3),flush=True)
        result=dict(backbone=name,batch=batch,step_seconds=time.time()-started,
                    peak_gb=torch.cuda.max_memory_allocated()/1e9)
        results.append(result);print(result,flush=True)
        del optimizer,x,p,target,logits,loss
    except Exception as exc:
        results.append(dict(backbone=name,error=repr(exc)));print(results[-1],flush=True)
    finally:
        del model;gc.collect();torch.cuda.empty_cache()
with open('/kaggle/working/gpu_preflight.json','w') as f:json.dump(results,f,indent=2)
